In [1]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np

In [2]:
# Defining adjacency list for given graph
A = {
    1:[2, 3, 4, 5], # Node 1
    2:[6, 7, 3], # Node 2
    5:[8, 9, 4], # Node 5
    3:[2, 4, 6, 7, 8], # Node 3
    4:[3, 5, 7, 8, 9], # Node 4
    6:[10, 11, 7], # Node 6
    9:[12, 13, 8], # Node 9
    7:[6, 8, 10, 11, 12], # Node 7
    8:[7, 9, 11, 12, 13], # Node 8
    10:[14, 15, 11], # Node 10
    13:[16, 17, 12], # Node 13
    11:[10, 12, 14, 15, 16], # Node 11
    12:[11, 13, 15, 16, 17], # Node 12
    14:[18, 15], # Node 14
    17:[18, 16], # Node 17
    15:[18, 14, 16], # Node 15
    16:[18, 15, 17], # Node 16
    18:[] # Node 18
}

V = list(A.keys())
E = [(i, j) for i in V for j in A[i]]
C = {(i, j): i + j for i, j in E}

print("Vertices:\n", V)
print("Edges::\n", E)
print("Cost:\n", C)

Vertices:
 [1, 2, 5, 3, 4, 6, 9, 7, 8, 10, 13, 11, 12, 14, 17, 15, 16, 18]
Edges::
 [(1, 2), (1, 3), (1, 4), (1, 5), (2, 6), (2, 7), (2, 3), (5, 8), (5, 9), (5, 4), (3, 2), (3, 4), (3, 6), (3, 7), (3, 8), (4, 3), (4, 5), (4, 7), (4, 8), (4, 9), (6, 10), (6, 11), (6, 7), (9, 12), (9, 13), (9, 8), (7, 6), (7, 8), (7, 10), (7, 11), (7, 12), (8, 7), (8, 9), (8, 11), (8, 12), (8, 13), (10, 14), (10, 15), (10, 11), (13, 16), (13, 17), (13, 12), (11, 10), (11, 12), (11, 14), (11, 15), (11, 16), (12, 11), (12, 13), (12, 15), (12, 16), (12, 17), (14, 18), (14, 15), (17, 18), (17, 16), (15, 18), (15, 14), (15, 16), (16, 18), (16, 15), (16, 17)]
Cost:
 {(1, 2): 3, (1, 3): 4, (1, 4): 5, (1, 5): 6, (2, 6): 8, (2, 7): 9, (2, 3): 5, (5, 8): 13, (5, 9): 14, (5, 4): 9, (3, 2): 5, (3, 4): 7, (3, 6): 9, (3, 7): 10, (3, 8): 11, (4, 3): 7, (4, 5): 9, (4, 7): 11, (4, 8): 12, (4, 9): 13, (6, 10): 16, (6, 11): 17, (6, 7): 13, (9, 12): 21, (9, 13): 22, (9, 8): 17, (7, 6): 13, (7, 8): 15, (7, 10): 17, (7, 11): 

#### Lower Level Shortest Path Problem

Solving the shortest path problem without interdiction in above network

In [ ]:
model = gp.Model("shortest_path_interdict")

# We drop the integrality in lower level problem as this does not affect SP solution. 
# Totally Unimodular property of the constraint matrix
x = model.addVars(E, vtype=GRB.CONTINUOUS, name="sp_edges") # follower's decision variables
y = model.addVars(E, vtype=GRB.BINARY, name="interdiction") # leader's decision variables

B = 3
s = 1
t = 18
b = {i:1 if i==s else -1 if i==t else 0 for i in V}

model.addConstrs(
    (
        gp.quicksum(x[u, v] for u, v in E if u == i)
        - gp.quicksum(x[u, v] for u, v in E if v == i)
        == b[i]
        for i in V
    ),
    name="flow_conservation"
)

model.setObjective(gp.quicksum(C[i, j] * x[i,j] for i, j in E), GRB.MINIMIZE)

model.optimize()

Set parameter Username
Set parameter LicenseID to value 2803608
Academic license - for non-commercial use only - expires 2027-04-05
Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 18 rows, 124 columns and 124 nonzeros (Min)
Model fingerprint: 0x79d04f7c
Model has 62 linear objective coefficients
Variable types: 62 continuous, 62 integer (62 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [3e+00, 4e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]

Found heuristic solution: objective 99.0000000
Presolve removed 0 rows and 62 columns
Presolve time: 0.00s
Presolved: 18 rows, 62 columns, 124 nonzeros
Variable types: 62 continuous, 0 integer (0 binary)

Root relaxation: objective 8.300000e+01, 14 iterations, 0.00 sec

In [4]:
print(f"Shortest Path Cost:", model.ObjVal)
print("\nSelected Shortest Paths Arcs:")
for i, j in E:
    if x[i, j].X == 1:
        print(f"{i}-{j}")

Shortest Path Cost: 83.0

Selected Shortest Paths Arcs:
1-2
2-6
6-10
10-14
14-18


In [5]:
model.reset()

Discarded solution information


#### 1. Interdiction in Objective

In [7]:
E_y = [(i, j) for i, j in E if i != 1 and j != 18]
E_protected = [e for e in E if e not in E_y]

Q = {(i, j): C[i, j] for i, j in E_y}

B = 3
s = 1
t = 18
b = {i: 1 if i == s else -1 if i == t else 0 for i in V}


In [ ]:

model_int_obj = gp.Model("objective_interdiction_strong_duality")

# Follower primal variables
x = model_int_obj.addVars(E, lb=0, ub=1, vtype=GRB.CONTINUOUS, name="sp_edges")

# Leader interdiction variables
y = model_int_obj.addVars(E_y, vtype=GRB.BINARY, name="interdiction")

# Follower dual variables
dual1 = model_int_obj.addVars(V, lb=-GRB.INFINITY, name="flow_conservation_dual")

# Upper-level interdiction budget
model_int_obj.addConstr(gp.quicksum(y[i, j] for i, j in E_y) <= B, name="interdiction_budget")

# Follower primal feasibility
model_int_obj.addConstrs((gp.quicksum(x[u, v] for u, v in E if u == i) - gp.quicksum(x[u, v] for u, v in E if v == i) 
                          == b[i] for i in V), name="flow_conservation")

# Follower dual feasibility
model_int_obj.addConstrs((dual1[i] - dual1[j] <= C[i, j] + Q[i, j]*y[i, j] for i, j in E_y), name="dual_interdicted")

model_int_obj.addConstrs((dual1[i] - dual1[j] <= C[i, j] for i, j in E_protected), name="dual_protected")

# Follower primal and dual objective expressions
primal_obj = gp.quicksum(C[i, j]*x[i, j] for i, j in E) + gp.quicksum(Q[i, j]*y[i, j]*x[i, j] for i, j in E_y)

dual_obj = dual1[s] - dual1[t]

# Strong duality constraint
model_int_obj.addConstr(primal_obj == dual_obj, name="strong_duality")

# Upper-level objective
model_int_obj.setObjective(primal_obj, GRB.MAXIMIZE)

model_int_obj.Params.NonConvex = 2
model_int_obj.optimize()


Set parameter NonConvex to value 2
Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Non-default parameters:
NonConvex  2

Optimize a model with 81 rows, 134 columns and 356 nonzeros (Max)
Model fingerprint: 0xdd09328b
Model has 62 linear objective coefficients
Model has 54 quadratic objective terms
Model has 1 quadratic constraint
Variable types: 80 continuous, 54 integer (54 binary)
Coefficient statistics:
  Matrix range     [1e+00, 3e+01]
  QMatrix range    [5e+00, 3e+01]
  QLMatrix range   [1e+00, 4e+01]
  Objective range  [3e+00, 4e+01]
  QObjective range [1e+01, 7e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 4e+01]

Presolve time: 0.00s
Presolved: 244 rows, 188 columns, 852 nonzeros
Variable types: 134 continuous, 54 integer (54 binary)
Found heuristic solution: objec

In [9]:
print(f"Shortest Path Cost:", model_int_obj.ObjVal)

print("\nInterdicted Shortest Paths Arcs:")
for i, j in E_y:
    if round(y[i, j].X) == 1:
        print(f"{i}-{j}")

print("\nSelected Shortest Paths Arcs:")
for i, j in E:
    if round(x[i, j].X) == 1:
        print(f"{i}-{j}")

Shortest Path Cost: 87.0

Interdicted Shortest Paths Arcs:
2-6
10-14

Selected Shortest Paths Arcs:
1-2
2-7
7-11
11-14
14-18


In [13]:
model_int_obj.reset()

Discarded solution information


#### Interdiction in Objective: Simplification

In [10]:
# Dual-only MILP formulation
model_int_obj2 = gp.Model("interdiction_in_objective")

# Leader interdiction variables
y = model_int_obj2.addVars(E_y, vtype=GRB.BINARY, name="interdiction")

# Follower dual variables
dual1 = model_int_obj2.addVars(V, lb=-GRB.INFINITY, name="flow_conservation_dual")

# Upper-level interdiction budget
model_int_obj2.addConstr(gp.quicksum(y[i, j] for i, j in E_y) <= B, name="interdiction_budget")

# Follower dual feasibility
model_int_obj2.addConstrs((dual1[i] - dual1[j] <= C[i, j] + Q[i, j]*y[i, j] for i, j in E_y), name="interdicted_edges")

model_int_obj2.addConstrs((dual1[i] - dual1[j] <= C[i, j] for i, j in E if (i, j) not in E_y), name="non_interdicted_edges")

# Upper-level objective
model_int_obj2.setObjective(dual1[s] - dual1[t], GRB.MAXIMIZE)

model_int_obj2.optimize()


Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 63 rows, 72 columns and 232 nonzeros (Max)
Model fingerprint: 0x9f4c6ba6
Model has 2 linear objective coefficients
Variable types: 18 continuous, 54 integer (54 binary)
Coefficient statistics:
  Matrix range     [1e+00, 3e+01]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [3e+00, 4e+01]

Found heuristic solution: objective 35.0000000
Presolve removed 1 rows and 2 columns
Presolve time: 0.00s
Presolved: 62 rows, 70 columns, 219 nonzeros
Variable types: 17 continuous, 53 integer (53 binary)

Root relaxation: objective 9.941872e+01, 49 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | 

In [11]:
print(f"Shortest Path Cost:", model_int_obj2.ObjVal)

print("\nInterdicted Shortest Paths Arcs:")
for i, j in E_y:
    if round(y[i, j].X) == 1:
        print(f"{i}-{j}")

print("\nSelected Shortest Paths Arcs:")
for i, j in E:
    if round(x[i, j].X) == 1:
        print(f"{i}-{j}")

Shortest Path Cost: 87.0

Interdicted Shortest Paths Arcs:
2-6
10-14

Selected Shortest Paths Arcs:
1-2
2-7
7-11
11-14
14-18


In [12]:
model_int_obj2.reset()

Discarded solution information


Same solution for simplification

#### 2. Solving interdiction problem Using the MILP reformulation with interdiction in constraints: Complementary slackness

In [ ]:
model_int_cons_comp_sla = gp.Model("model_milp2")

x = model_int_cons_comp_sla.addVars(E, vtype=GRB.CONTINUOUS, name="sp_edges")
y = model_int_cons_comp_sla.addVars(E_y, vtype=GRB.BINARY, name="interdiction")

# Dual variables of follower Shortest path problem
dual1 = model_int_cons_comp_sla.addVars(V, lb=-GRB.INFINITY, name="flow_conservation_dual")
dual2 = model_int_cons_comp_sla.addVars(E, lb=-GRB.INFINITY, ub=0, name="interdiction_dual")

slack = model_int_cons_comp_sla.addVars(E, lb=0, name="slack")
slack_dual = model_int_cons_comp_sla.addVars(E, lb=0, name="slack_dual")

z1 = model_int_cons_comp_sla.addVars(E, vtype=GRB.BINARY, name="z1")
z2 = model_int_cons_comp_sla.addVars(E, vtype=GRB.BINARY, name="z2")

M = 100 # arbitrarily large number chosen for big-M method

# Interdiction budget constraint: Upper level constraint
model_int_cons_comp_sla.addConstr(gp.quicksum(y[i, j] for i, j in E_y) <= B, name="interdiction_budget")

# Followers Primal feasibility
model_int_cons_comp_sla.addConstrs(
    (
        gp.quicksum(x[u, v] for u, v in E if u == i)
        - gp.quicksum(x[u, v] for u, v in E if v == i)
        == b[i]
        for i in V
    ),
    name="flow_conservation"
)

# Interdiction constraints
model_int_cons_comp_sla.addConstrs((slack[i, j] == 1 - y[i, j] - x[i, j] for i, j in E_y), name="interdiction_constraint")
model_int_cons_comp_sla.addConstrs((slack[i, j] == 1 - x[i, j] for i, j in E if (i, j) not in E_y), name="non_interdicted_edges")

# Followers Dual feasibility
model_int_cons_comp_sla.addConstrs((
        slack_dual[i, j] ==
        -dual1[i] + dual1[j] - dual2[i, j] + C[i, j]
        for i, j in E),
    name="dual_constraint"
)
# Complementary slackness
model_int_cons_comp_sla.addConstrs((-dual2[i, j] <= M * z1[i, j] for i, j in E), name="CS1_dual")
model_int_cons_comp_sla.addConstrs((slack[i, j] <= M * (1-z1[i, j]) for i, j in E), name="CS1_primal")

# Complementary slackness
model_int_cons_comp_sla.addConstrs((slack_dual[i, j] <= M * z2[i, j] for i, j in E),name="CS2_dual")
model_int_cons_comp_sla.addConstrs(x[i, j] <= M * (1-z2[i, j]) for i, j in E)

model_int_cons_comp_sla.setObjective(gp.quicksum(C[i, j] * x[i,j] for i, j in E), GRB.MAXIMIZE)
model_int_cons_comp_sla.optimize()


Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 391 rows, 444 columns and 1100 nonzeros (Max)
Model fingerprint: 0xaf985c26
Model has 62 linear objective coefficients
Variable types: 266 continuous, 178 integer (178 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+02]
  Objective range  [3e+00, 4e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+02]

Presolve removed 8 rows and 9 columns
Presolve time: 0.00s
Presolved: 383 rows, 435 columns, 1074 nonzeros
Variable types: 257 continuous, 178 integer (178 binary)
Found heuristic solution: objective 83.0000000

Root relaxation: objective 4.564800e+02, 475 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  D

In [ ]:
print(f"Shortest Path Cost:", model_int_cons_comp_sla.ObjVal)

print("\nInterdicted Shortest Paths Arcs:")
for i, j in E_y:
    if round(y[i, j].X) == 1:
        print(f"{i}-{j}")

print("\nSelected Shortest Paths Arcs:")
for i, j in E:
    if round(x[i, j].X) == 1:
        print(f"{i}-{j}")

Shortest Path Cost: 89.00013849093031

Interdicted Shortest Paths Arcs:
2-6
4-6

Selected Shortest Paths Arcs:
1-3
3-8
8-10
10-14
14-18


#### 3. Using Strong Duality


In [ ]:

model_SD = gp.Model("model_strong_duality")

x = model_SD.addVars(E, vtype=GRB.CONTINUOUS, name="sp_edges")

E_y = [(i, j) for i, j in E if i != 1 and j != 18]
E_protected = [e for e in E if e not in E_y]
y = model_SD.addVars(E_y, vtype=GRB.BINARY, name="interdiction")

# Dual variables of follower Shortest Path problem
dual1 = model_SD.addVars(V, lb=-GRB.INFINITY, name="flow_conservation_dual")
dual2 = model_SD.addVars(E, lb=-GRB.INFINITY, ub=0, name="interdiction_dual")

# Interdiction budget constraint: Upper level
model_SD.addConstr(gp.quicksum(y[i, j] for i, j in E_y) <= B, name="interdiction_budget")

# Follower's primal feasibility: Flow conservation
model_SD.addConstrs((gp.quicksum(x[u, v] for u, v in E if u == i) - gp.quicksum(x[u, v] for u, v in E if v == i) 
                   == b[i] for i in V), name="flow_conservation")

# Follower's primal feasibility: Interdiction constraints
model_SD.addConstrs((x[i, j] <= 1 - y[i, j] for i, j in E_y), name="interdiction_constraint")
model_SD.addConstrs((x[i, j] <= 1 for i, j in E_protected), name="non_interdicted_edges")

# Follower's dual feasibility
model_SD.addConstrs((dual1[i] - dual1[j] + dual2[i, j] <= C[i, j] for i, j in E), name="dual_constraint")

# Primal and dual objective
primal_obj = gp.quicksum(C[i, j] * x[i, j] for i, j in E)
dual_obj = (dual1[s] - dual1[t] + gp.quicksum((1 - y[i, j]) * dual2[i, j] for i, j in E_y) + gp.quicksum(dual2[i, j] for i, j in E_protected))
model_SD.addConstr(primal_obj == dual_obj, name="strong_duality") # Strong duality constraint

model_SD.setObjective(primal_obj, GRB.MAXIMIZE)
model_SD.Params.NonConvex = 2
model_SD.optimize()


Set parameter NonConvex to value 2
Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Non-default parameters:
NonConvex  2

Optimize a model with 143 rows, 196 columns and 480 nonzeros (Max)
Model fingerprint: 0xe627f018
Model has 62 linear objective coefficients
Model has 1 quadratic constraint
Variable types: 142 continuous, 54 integer (54 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  QMatrix range    [1e+00, 1e+00]
  QLMatrix range   [1e+00, 4e+01]
  Objective range  [3e+00, 4e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 4e+01]

Presolve removed 8 rows and 0 columns
Presolve time: 0.00s
Presolved: 244 rows, 358 columns, 922 nonzeros
Presolved model has 108 SOS constraint(s)
Variable types: 250 continuous, 108 integer (108 binary)

Root relaxation: ob

Shortest Paths Solution

In [ ]:
print(f"Shortest Path Cost:", model_SD.ObjVal)

print("\nInterdicted Shortest Paths Arcs:")
for i, j in E_y:
    if round(y[i, j].X) == 1:
        print(f"{i}-{j}")

print("\nSelected Shortest Paths Arcs:")
for i, j in E:
    if round(x[i, j].X) == 1:
        print(f"{i}-{j}")

Shortest Path Cost: 87.0

Interdicted Shortest Paths Arcs:
2-6
2-7
3-6

Selected Shortest Paths Arcs:
1-3
3-7
7-10
10-14
14-18


#### 4. Using Branch and Bound in the same reformulation (SOS1)

In [ ]:
model_bb = gp.Model("model_milp2")

x = model_bb.addVars(E, lb=0, ub=1, vtype=GRB.CONTINUOUS, name="sp_edges")
y = model_bb.addVars(E_y, vtype=GRB.BINARY, name="interdiction")

# Dual variables of follower Shortest path problem
dual1 = model_bb.addVars(V, lb=-GRB.INFINITY, name="flow_conservation_dual")
dual2 = model_bb.addVars(E, lb=-GRB.INFINITY, ub=0, name="interdiction_dual")

# Slack variable for interdiction constraints
slack = model_bb.addVars(E, lb=0, name="slack")

# Slack variable for dual constraints
slack_dual = model_bb.addVars(E, lb=0, name="slack_dual")


# Interdiction budget constraint: Upper level constraint
model_bb.addConstr(gp.quicksum(y[i, j] for i, j in E_y) <= B, name="interdiction_budget")

# Followers Primal feasibility
model_bb.addConstrs((gp.quicksum(x[u, v] for u, v in E if u == i) - gp.quicksum(x[u, v] for u, v in E if v == i) 
                   == b[i] for i in V), name="flow_conservation")

# Interdiction constraints
model_bb.addConstrs((slack[i, j] == 1 - y[i, j] - x[i, j] for i, j in E_y), name="interdiction_constraint")
model_bb.addConstrs((slack[i, j] == 1 - x[i, j] for i, j in E if (i, j) not in E_y), name="non_interdicted_edges")

# Followers Dual feasibility
model_bb.addConstrs((slack_dual[i, j] == -dual1[i] + dual1[j] - dual2[i, j] + C[i, j] for i, j in E), name="dual_constraint")

# Complementary slackness using SOS1
for i, j in E:
    model_bb.addSOS(GRB.SOS_TYPE1, [dual2[i, j], slack[i, j]])
    model_bb.addSOS(GRB.SOS_TYPE1, [x[i, j], slack_dual[i, j]])

model_bb.setObjective(gp.quicksum(C[i, j] * x[i,j] for i, j in E), GRB.MAXIMIZE)
model_bb.Params.TimeLimit = 1200
model_bb.Params.MIPGap = 0.01
model_bb.optimize()

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 143 rows, 320 columns and 604 nonzeros (Max)
Model fingerprint: 0x1c0be878
Model has 62 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [3e+00, 4e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 4e+01]

Presolve removed 126 rows and 258 columns
Presolve time: 0.01s
Presolved: 17 rows, 62 columns, 114 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    1.1660000e+03   1.800000e+01   0.000000e+00      0s

Extra simplex iterations after uncrush: 17
      35    5.6300000e+02   0.000000e+00   0.000000e+00      0s

Solved in 35 iterations and 0.01 seconds (0.00 work units)
Optimal objective  5.630000000e+02


In [ ]:
print(f"Shortest Path Cost:", model_bb.ObjVal)

print("\nInterdicted Shortest Paths Arcs:")
for i, j in E_y:
    if round(y[i, j].X) == 1:
        print(f"{i}-{j}")

print("\nSelected Shortest Paths Arcs:")
for i, j in E:
    if round(x[i, j].X) == 1:
        print(f"{i}-{j}")